# SPLADE, ColBERT and Ensemble Retrieval: Practical Demo

We demonstrate three retrieval techniques on a short text about **Artificial Intelligence in healthcare**, with real models and a real LLM (Groq), using LangChain.

1. **SPLADE (learned sparse retrieval):** keyword search, but a neural model first **expands** the words with related words.
2. **ColBERT retrieval:** one embedding **per token** (not per chunk), with token-level matching.
3. **Ensemble retrieval:** run several retrievers (SPLADE + dense + ColBERT) and merge their rankings by **voting**.

For every technique we run the full RAG flow: **retrieve → augment → generate**.

**Important:** our text is short (only 6 sentences = 6 chunks). So instead of only asking "is the right chunk in the top 2?", we measure the **rank of the right chunk (1 = best, 6 = worst)**.

## 1. Setup

In [1]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [2]:
pip install langchain-groq -q # q for silent installation

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

In [4]:
# langchain-huggingface : dense embeddings
# langchain-qdrant      : vector store that also supports sparse (SPLADE) vectors
# fastembed             : runs SPLADE and ColBERT models on CPU
!pip install langchain-huggingface langchain-qdrant fastembed -q

## 2. Our text, chunks, questions and helpers

### 2.1 The text and its chunks

We split the text into **one chunk per sentence**.

In [5]:
import re

text = """Artificial Intelligence (AI) is transforming healthcare by improving disease diagnosis, patient monitoring, and treatment planning. Machine learning models analyze medical records, laboratory reports, and medical images to identify patterns and predict potential health risks.

Deep learning techniques, such as Convolutional Neural Networks (CNNs), are widely used for medical image classification. Natural Language Processing (NLP) helps extract meaningful information from clinical notes and electronic health records.

However, AI adoption in healthcare faces challenges, including data privacy, model bias, high implementation costs, and the need for reliable predictions. Combining AI technologies with human expertise can improve healthcare efficiency and patient outcomes."""

# Split after . ! ? followed by whitespace (our text has no abbreviations like "Dr.")
chunks = re.split(r'(?<=[.!?])\s+', text)

for i, chunk in enumerate(chunks):
    print(i, '|', chunk)

0 | Artificial Intelligence (AI) is transforming healthcare by improving disease diagnosis, patient monitoring, and treatment planning.
1 | Machine learning models analyze medical records, laboratory reports, and medical images to identify patterns and predict potential health risks.
2 | Deep learning techniques, such as Convolutional Neural Networks (CNNs), are widely used for medical image classification.
3 | Natural Language Processing (NLP) helps extract meaningful information from clinical notes and electronic health records.
4 | However, AI adoption in healthcare faces challenges, including data privacy, model bias, high implementation costs, and the need for reliable predictions.
5 | Combining AI technologies with human expertise can improve healthcare efficiency and patient outcomes.


Vector stores in LangChain store **Document** objects. We keep the chunk number in the metadata, so we can find each chunk again after a search.

In [6]:
from langchain_core.documents import Document

docs = [Document(page_content=chunk, metadata={'id': i}) for i, chunk in enumerate(chunks)]

### 2.2 Three test questions

Each question is written to test a strength or weakness. `keyword` is only used to find the chunk that contains the right answer.

| Question | Right chunk | What it tests |
|---|---|---|
| What are CNNs used for? | CNN sentence | exact term from the text |
| What obstacles stop hospitals from using this technology? | challenges sentence | paraphrase: almost no shared words |
| How can computers understand what doctors write in patient files? | NLP sentence | meaning without matching words |

In [7]:
import numpy as np
import pandas as pd

# (label, question, keyword found only in the right chunk)
queries = [
    ('exact term', 'What are CNNs used for?', 'Convolutional'),
    ('paraphrase', 'What obstacles stop hospitals from using this technology?', 'challenges'),
    ('no shared words', 'How can computers understand what doctors write in patient files?', 'Natural Language Processing'),
]

# Position (index) of the right chunk for every question
right_ids = [next(i for i, chunk in enumerate(chunks) if keyword in chunk) for _, _, keyword in queries]
print('Right chunk ids:', right_ids)

Right chunk ids: [2, 4, 3]


### 2.3 Helper functions

Every retriever below is a function that gives **one score per chunk** (higher = more relevant). `run(scores_fn)` does the full RAG flow with any of them.

In [8]:
# Augmentation + Generation
def ask(question, retrieved_chunks):
    # Augmentation: combine the retrieved chunks with the user question
    context = '\n\n'.join(retrieved_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""
    # Generation: LLM answers on the basis of the retrieved info
    return llm.invoke(prompt).content

In [9]:
# Position of the right chunk when all chunks are sorted by score (1 = best)
def rank_of(scores, right_id):
    order = np.argsort(-scores, kind='stable')   # best chunk first (ties keep the original order)
    return list(order).index(right_id) + 1

In [10]:
# Vector stores return (document, score) pairs: turn them into one score per chunk
# (chunks that the store does not return get score 0)
def to_scores(results):
    scores = np.zeros(len(chunks))
    for doc, score in results:
        scores[doc.metadata['id']] = score
    return scores

In [11]:
# Full RAG flow for all questions with any retriever
def run(scores_fn):
    for (label, question, _), right_id in zip(queries, right_ids):
        scores = scores_fn(question)                    # Retrieval: one score per chunk
        top_ids = np.argsort(-scores, kind='stable')[:2]
        print(f'QUESTION ({label}):', question)
        print('  Scores of all chunks:', scores.round(2))
        for rank, i in enumerate(top_ids, start=1):
            print(f'  {rank}. chunk {i}: {chunks[i][:80]}...')
        print('  Rank of the right chunk:', rank_of(scores, right_id))
        print('  ANSWER:', ask(question, [chunks[i] for i in top_ids]))
        print()

## 3. SPLADE (learned sparse retrieval)

**BM25** sees the word *car* and searches for *car*. **SPLADE** uses a neural model to **expand** each word: it sees *car* and also searches for *automobile, vehicle, driving*, etc. The result is still a **sparse vector** (one weight per vocabulary word, mostly zeros), so it is keyword search, but with some understanding of meaning.

- **Advantage:** keeps the speed and interpretability of sparse search, but adds semantic expansion. Often better than BM25 and pure dense retrieval for keyword-heavy queries, and very common inside hybrid retrieval.
- **Disadvantage:** needs a neural model to encode every chunk and query (slower than BM25), and expanded vectors have many more non-zero terms than the original text has words.

### 3.1 Load the SPLADE model and store the chunks

In [12]:
from langchain_qdrant import FastEmbedSparse

# A real SPLADE model
sparse_embeddings = FastEmbedSparse(model_name='prithivida/Splade_PP_en_v1')

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



In [14]:
from langchain_qdrant import QdrantVectorStore, RetrievalMode

# Every chunk is converted into a sparse SPLADE vector and stored (in memory)
splade_store = QdrantVectorStore.from_documents(
    docs,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    location=':memory:',
    collection_name='splade_chunks',
    retrieval_mode=RetrievalMode.SPARSE
)

### 3.2 See the expansion

We encode the paraphrased question and print the words with the highest weights. Compare them with the words of the original question.

In [15]:
from transformers import AutoTokenizer

# SPLADE vectors have one position per word of the BERT vocabulary
vocab_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

question = queries[1][1]
query_vec = sparse_embeddings.embed_query(question)

# The 12 words with the highest weights
top = sorted(zip(query_vec.values, query_vec.indices), reverse=True)[:12]
print('Question:', question)
print('Top SPLADE terms:', [(vocab_tokenizer.convert_ids_to_tokens(int(i)), round(float(v), 2)) for v, i in top])

Question: What obstacles stop hospitals from using this technology?
Top SPLADE terms: [('technology', 2.26), ('obstacles', 2.21), ('hospital', 2.02), ('obstacle', 2.01), ('hospitals', 1.9), ('prevent', 1.51), ('use', 1.41), ('stop', 1.37), ('technologies', 1.36), ('barriers', 1.24), ('medical', 1.07), ('fight', 0.78)]


In [16]:
chunk_vecs = sparse_embeddings.embed_documents(chunks)

print('Words in the question:', len(question.split()), '| non-zero terms in its SPLADE vector:', len(query_vec.indices))
print('Average words per chunk:', round(np.mean([len(c.split()) for c in chunks])),
      '| average non-zero terms per chunk vector:', round(np.mean([len(v.indices) for v in chunk_vecs])))

Words in the question: 8 | non-zero terms in its SPLADE vector: 52
Average words per chunk: 16 | average non-zero terms per chunk vector: 106


### 3.3 Retrieve → Augment → Generate with SPLADE

In [17]:
def splade_scores(question):
    # Score = dot product of the sparse query vector and each sparse chunk vector
    return to_scores(splade_store.similarity_search_with_score(question, k=len(chunks)))

run(splade_scores)

QUESTION (exact term): What are CNNs used for?
  Scores of all chunks: [ 1.96  1.25 18.82  1.44  1.02  1.59]
  1. chunk 2: Deep learning techniques, such as Convolutional Neural Networks (CNNs), are wide...
  2. chunk 0: Artificial Intelligence (AI) is transforming healthcare by improving disease dia...
  Rank of the right chunk: 1
  ANSWER: Medical image classification.

QUESTION (paraphrase): What obstacles stop hospitals from using this technology?
  Scores of all chunks: [ 4.99  3.08  5.26  3.55  7.49 10.58]
  1. chunk 5: Combining AI technologies with human expertise can improve healthcare efficiency...
  2. chunk 4: However, AI adoption in healthcare faces challenges, including data privacy, mod...
  Rank of the right chunk: 2
  ANSWER: Data privacy, model bias, high implementation costs, and the need for reliable predictions.

QUESTION (no shared words): How can computers understand what doctors write in patient files?
  Scores of all chunks: [5.45 3.06 2.46 3.38 2.93 6.19]
  1.

**How to read this:** check the expanded terms above, and the rank of the right chunk for each question. For the paraphrased question, see whether SPLADE found the right chunk even though the question shares few exact words with it. Chunks with no matching term are not returned by a sparse search, so they get score 0.

## 4. ColBERT retrieval

Normal (dense) retrieval squeezes a whole chunk into **one vector**. ColBERT keeps **one vector per token**. At search time, every query token looks for its best-matching token in the chunk (**MaxSim**), and these best matches are added up:

```
score(query, chunk) = sum over query tokens of ( max similarity with any chunk token )
```

- **Advantage:** very precise matching, because it compares at token level.
- **Disadvantage:** needs far more storage (one vector per token) and more computation.
- **Best when** precision really matters (legal, financial documents) and you can afford the storage.

In [18]:
from fastembed import LateInteractionTextEmbedding

# A real ColBERTv2 model
colbert = LateInteractionTextEmbedding('colbert-ir/colbertv2.0')

# One embedding per TOKEN for every chunk
doc_embs = list(colbert.embed(chunks))

print('Number of chunks:', len(chunks))
print('Number of vectors stored:', sum(len(d) for d in doc_embs))  # dense retrieval would store one per chunk

Number of chunks: 6
Number of vectors stored: 123


In [19]:
def colbert_scores(question):
    q = list(colbert.query_embed(question))[0]   # one vector per query token
    # MaxSim: for each query token take its best match in the chunk, then add them up
    return np.array([(q @ d.T).max(axis=1).sum() for d in doc_embs])

run(colbert_scores)

QUESTION (exact term): What are CNNs used for?
  Scores of all chunks: [ 8.45  8.9  25.76 11.03  4.59  6.39]
  1. chunk 2: Deep learning techniques, such as Convolutional Neural Networks (CNNs), are wide...
  2. chunk 3: Natural Language Processing (NLP) helps extract meaningful information from clin...
  Rank of the right chunk: 1
  ANSWER: Medical image classification.

QUESTION (paraphrase): What obstacles stop hospitals from using this technology?
  Scores of all chunks: [12.07 10.45  9.93 10.1  16.65 15.35]
  1. chunk 4: However, AI adoption in healthcare faces challenges, including data privacy, mod...
  2. chunk 5: Combining AI technologies with human expertise can improve healthcare efficiency...
  Rank of the right chunk: 1
  ANSWER: Data privacy, model bias, high implementation costs, and the need for reliable predictions.

QUESTION (no shared words): How can computers understand what doctors write in patient files?
  Scores of all chunks: [11.61 14.71 11.48 14.8   8.25 10.49

## 5. Ensemble retrieval

Run **several different retrievers** and merge their ranked lists. We use **Reciprocal Rank Fusion (RRF)**, a voting rule: every retriever gives each chunk the points `1 / (60 + rank)`, and the points are added up.

A chunk that is ranked high by **several** retrievers wins. If all retrievers agree, the chunk is almost certainly relevant. If only one retriever likes it, we are less sure.

- **Advantage:** high recall, so it is unlikely to miss relevant chunks; one retriever's blind spot is covered by the others.
- **Disadvantage:** runs several retrievers, so it costs more compute and time.

Our three retrievers: **SPLADE** (expanded keywords), **dense** (meaning, one vector per chunk) and **ColBERT** (token-level).

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Dense retriever: one embedding per chunk, compared by cosine similarity
dense_embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
dense_store = InMemoryVectorStore.from_documents(docs, embedding=dense_embeddings)

def dense_scores(question):
    return to_scores(dense_store.similarity_search_with_score(question, k=len(chunks)))

In [21]:
def ensemble_scores(question, k=60):
    fused = np.zeros(len(chunks))
    for scores_fn in (splade_scores, dense_scores, colbert_scores):
        ranking = np.argsort(-scores_fn(question), kind='stable')   # chunks ranked best first
        for rank, i in enumerate(ranking, start=1):
            fused[i] += 1 / (k + rank)                              # Reciprocal Rank Fusion
    return fused

run(ensemble_scores)

QUESTION (exact term): What are CNNs used for?
  Scores of all chunks: [0.05 0.05 0.05 0.05 0.05 0.05]
  1. chunk 2: Deep learning techniques, such as Convolutional Neural Networks (CNNs), are wide...
  2. chunk 0: Artificial Intelligence (AI) is transforming healthcare by improving disease dia...
  Rank of the right chunk: 1
  ANSWER: Medical image classification.

QUESTION (paraphrase): What obstacles stop hospitals from using this technology?
  Scores of all chunks: [0.05 0.05 0.05 0.05 0.05 0.05]
  1. chunk 4: However, AI adoption in healthcare faces challenges, including data privacy, mod...
  2. chunk 5: Combining AI technologies with human expertise can improve healthcare efficiency...
  Rank of the right chunk: 1
  ANSWER: Data privacy, model bias, high implementation costs, and the need for reliable predictions.

QUESTION (no shared words): How can computers understand what doctors write in patient files?
  Scores of all chunks: [0.05 0.05 0.05 0.05 0.05 0.05]
  1. chunk 3: Na

## 6. Comparison

Rank of the **right chunk** for each retriever and question (1 = best, 6 = worst).

In [22]:
methods = {'SPLADE': splade_scores, 'Dense': dense_scores, 'ColBERT': colbert_scores, 'Ensemble': ensemble_scores}

pd.DataFrame({
    name: [rank_of(fn(question), right_id) for (_, question, _), right_id in zip(queries, right_ids)]
    for name, fn in methods.items()
}, index=[label for label, _, _ in queries])

,SPLADE,Dense,ColBERT,Ensemble
exact term,1,1,1,1
paraphrase,2,1,1,1
no shared words,3,1,1,1


## 7. Summary

| Technique | Advantage | Disadvantage | Use when |
|---|---|---|---|
| **SPLADE** | BM25-level speed and interpretability plus semantic expansion (car → automobile, vehicle) | Needs a neural model to encode chunks and queries; more non-zero terms than BM25 | Keyword-heavy queries where synonyms matter; inside hybrid retrieval |
| **ColBERT** | Very precise token-level matching | Much more storage (one vector per token); more computation | Precision really matters (legal, financial) and storage is affordable |
| **Ensemble** | High recall; covers each retriever's blind spots; agreement between retrievers means high confidence | Runs several retrievers: more compute and complexity | You cannot afford to miss relevant chunks |